In [ ]:
import pandas as pd
DIR = r'../data/clean/labeled_data.csv'

data = pd.read_csv(DIR, index_col=0)[['label']]

FROM_DIR = r'../data/tickers.csv'
tickers=pd.read_csv(FROM_DIR, index_col=1)['Ticker']

data = pd.merge(data, tickers, how='left', left_index=True, right_index=True)
data.loc['CCC', 'Ticker'] = 'MDV'
print(data)

In [ ]:
portfelA = data[data['label']=='A']
a_value = pd.DataFrame()
n_A = portfelA.shape[0]
portfelA['liczba akcji'] = 100000/n_A / portfelA['price1']

portfelB = data[data['label']=='B']
b_value = pd.DataFrame()
n_B = portfelB.shape[0]
portfelB['liczba akcji'] = 100000/n_B / portfelB['price1']

portfelC = data[data['label']=='C']
c_value = pd.DataFrame()
n_C = portfelC.shape[0]
portfelC['liczba akcji'] = 100000/n_C / portfelC['price1']
print(portfelB)

In [ ]:
from src.indicators import get_share_price
errors=0
i=1
price_jan = {}
price_dec = {}
std = {}
for ticker, label in zip(data['Ticker'], data['label']):
    try:
        print(f'Pobieranie {i}/30')
        share_price_series = get_share_price(ticker=ticker)
        share_price_2025 = share_price_series[share_price_series['rok'] == 2025]['Zamkniecie']
        
        if label == 'A':
            a_value[ticker]=share_price_2025
        elif label == 'B':
            b_value[ticker]=share_price_2025
        else:
            c_value[ticker]=share_price_2025

        price_jan[ticker] = share_price_2025.iloc[0]
        price_dec[ticker] = share_price_2025.iloc[-1]
        std[ticker] = share_price_2025.pct_change().std()
    except Exception as e:
        print(f'Bład dla {ticker}: {e}')
        errors+=1
    i+=1

print(f"Liczba bledow: {errors}")


In [ ]:
wig = get_share_price('wig').query('rok == 2025')['Zamkniecie']
print(wig.info())
wig.to_csv(r'../data/clean/wig.csv')

In [ ]:
print(a_value.info())
print(b_value.info())
print(c_value.info())

a_info = portfelA['liczba akcji'].values
a_value = a_value*a_info

b_info = portfelB['liczba akcji'].values
b_value = b_value*b_info

c_info = portfelC['liczba akcji'].values
c_value = c_value*c_info

In [ ]:
a_tickers = portfelA['Ticker'].values
a_value['value'] = a_value[a_tickers].sum(axis=1)

In [ ]:
b_tickers = portfelB['Ticker'].values
b_value['value'] = b_value[b_tickers].sum(axis=1)

In [ ]:
c_tickers = portfelC['Ticker'].values
c_value['value'] = c_value[c_tickers].sum(axis=1)
print(c_value)

In [ ]:
a_value.to_csv(r'../data/clean/a_value.csv')
b_value.to_csv(r'../data/clean/b_value.csv')
c_value.to_csv(r'../data/clean/c_value.csv')

In [ ]:
to_merge = pd.DataFrame([price_jan, price_dec, std], index=['price1','price2','std']).transpose()
print(to_merge.head())
print(to_merge.info())

In [ ]:
data = pd.merge(data, to_merge, left_on='Ticker', right_index=True)

In [ ]:
from src.indicators import get_dividend_per_share
dps_dict = {}
for comp in data.index:
    dps = get_dividend_per_share(comp).loc[2024]
    dps_dict[comp] = dps

data['dps']=dps_dict
print(data.head())


In [ ]:
data['z/s dyw'] = (data['price2']+data['dps']-data['price1'])/data['price1']
print(data.head())

TO_DIR = r'../data/clean/stock_prices.csv'
data.to_csv(TO_DIR)